In [1]:
import os
import pandas as pd
import numpy as np

# Parámetros de configuración:
annotations_dir = r"C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\annotations"
output_dir = r"C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\labels"
distance_threshold = 50      # Umbral en píxeles para considerar que un jugador "tiene control" del balón
min_frames_confirm = 3       # Número mínimo de frames consecutivos para confirmar un cambio en posesión

# Crea la carpeta de salida si no existe
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def compute_center(bb_left, bb_top, bb_width, bb_height):
    """
    Calcula el centro de una caja delimitadora.
    """
    center_x = bb_left + bb_width / 2
    center_y = bb_top + bb_height / 2
    return center_x, center_y

def smooth_possession(possession_list, min_frames):
    """
    Suaviza la secuencia de posesión para evitar cambios aislados.
    Si un cambio en la posesión dura menos de 'min_frames', se conserva la posesión previa.
    """
    smoothed = possession_list.copy()
    n = len(smoothed)
    # Primera pasada: corregir cambios aislados (si el frame actual difiere y sus vecinos son iguales)
    for i in range(1, n - 1):
        if smoothed[i] != smoothed[i - 1] and smoothed[i - 1] == smoothed[i + 1]:
            smoothed[i] = smoothed[i - 1]
    # Segunda pasada: garantizar que los cambios duren al menos 'min_frames'
    i = 0
    while i < n:
        current = smoothed[i]
        j = i
        while j < n and smoothed[j] == current:
            j += 1
        duration = j - i
        if duration < min_frames and i > 0:
            # Revertir cambio corto: asignar el valor anterior a los frames breves
            for k in range(i, j):
                smoothed[k] = smoothed[i - 1]
        i = j
    return smoothed

# Procesar cada archivo CSV en la carpeta de anotaciones
for filename in os.listdir(annotations_dir):
    if filename.endswith(".csv"):
        csv_path = os.path.join(annotations_dir, filename)
        print("Procesando:", csv_path)
        # Intentar detectar el delimitador automáticamente:
        df = pd.read_csv(csv_path, sep=None, engine='python', header=None)
        
        # Las primeras tres filas contienen metadatos. La primera columna de cada fila es la etiqueta,
        # así que la descartamos.
        team_row = df.iloc[0, 1:]
        playerid_row = df.iloc[1, 1:]
        attributes_row = df.iloc[2, 1:]
        
        # La fila 3 contiene el encabezado "frame", así que los datos reales comienzan en la fila 4.
        # Además, descartamos la primera columna de cada fila (que suele ser el número de frame).
        data = df.iloc[4:, 1:].reset_index(drop=True)
        data = data.astype(float)
        
        # Cada objeto (jugador o balón) ocupa 4 columnas: [bb_height, bb_left, bb_top, bb_width]
        group_size = 4
        num_cols = data.shape[1]
        num_objects = num_cols // group_size
        
        # Identificar cuál grupo corresponde al balón (según TeamID)
        ball_group_index = None
        player_groups = []
        for i in range(num_objects):
            col_index = i * group_size
            team_val = str(team_row.iloc[col_index]).strip().upper()
            if team_val == "BALL":
                ball_group_index = i
            else:
                player_groups.append(i)
        
        if ball_group_index is None:
            print("No se encontró el grupo del balón en", filename)
            continue
        
        print("Se encontraron", len(player_groups), "grupos de jugadores y el grupo del balón está en el índice", ball_group_index)
        
        # Calcular la posesión para cada frame
        raw_possession = []  # Almacena la posesión (0 o 1) por frame
        for idx, row in data.iterrows():
            # Obtener la caja del balón
            ball_vals = row[ball_group_index * group_size : (ball_group_index + 1) * group_size].values
            ball_bb_height, ball_bb_left, ball_bb_top, ball_bb_width = ball_vals
            ball_center = compute_center(ball_bb_left, ball_bb_top, ball_bb_width, ball_bb_height)
            
            # Calcular la distancia desde el balón a cada jugador
            min_dist = float("inf")
            possessing_team = None
            for i in player_groups:
                player_vals = row[i * group_size : (i + 1) * group_size].values
                if np.isnan(player_vals).any():
                    continue
                bb_height, bb_left, bb_top, bb_width = player_vals
                player_center = compute_center(bb_left, bb_top, bb_width, bb_height)
                dist = np.sqrt((player_center[0] - ball_center[0]) ** 2 + (player_center[1] - ball_center[1]) ** 2)
                if dist < min_dist:
                    min_dist = dist
                    try:
                        possessing_team = int(team_row.iloc[i * group_size])
                    except:
                        possessing_team = None
            
            # Si la distancia mínima excede el umbral, se asume que no hay control claro y se conserva la posesión anterior
            if min_dist > distance_threshold:
                if raw_possession:
                    raw_possession.append(raw_possession[-1])
                else:
                    raw_possession.append(None)
            else:
                raw_possession.append(possessing_team)
        
        # Suavizar la secuencia para evitar cambios bruscos
        smoothed_possession = smooth_possession(raw_possession, min_frames_confirm)
        
        # Crear un DataFrame de salida con el número de frame y la posesión
        frames = list(range(1, len(smoothed_possession) + 1))
        out_df = pd.DataFrame({
            "frame": frames,
            "team_in_possession": smoothed_possession
        })
        
        # Guardar el CSV de etiquetas en la carpeta de salida
        output_path = os.path.join(output_dir, "posesion_" + filename)
        out_df.to_csv(output_path, index=False, sep=';')
        print("Etiquetas guardadas en:", output_path)


Procesando: C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\annotations\D_20220220_1_0000_0030.csv
Se encontraron 22 grupos de jugadores y el grupo del balón está en el índice 22
Etiquetas guardadas en: C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\labels\posesion_D_20220220_1_0000_0030.csv
Procesando: C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\annotations\D_20220220_1_0030_0060.csv
Se encontraron 22 grupos de jugadores y el grupo del balón está en el índice 22
Etiquetas guardadas en: C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\labels\posesion_D_20220220_1_0030_0060.csv
Procesando: C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\annotations\D_20220220_1_0060_0090.csv
Se encontraron 22 grupos de jugadores y el grupo del balón está en el índice 22
Etiquetas guardadas en: C:\Users\ANDRES\Desktop\Master IA\ASIGNATURA

In [2]:
print(f"Ruta: {output_path}")

Ruta: C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\labels\posesion_D_20220220_1_1770_1800.csv


In [3]:
import cv2
import os
import pandas as pd
import os, sys


# Rutas de entrada/salida:
videos_dir = r"C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\videos"
labels_dir = r"C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\labels"
frames_output_dir = r"C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\frames"

# ────────────────────────────────────────────────
#  SKIP SI YA PROCESADO
# ────────────────────────────────────────────────
# (Asume que videos_dir y frames_output_dir ya están definidos más abajo)
video_files = [f for f in os.listdir(videos_dir) if f.endswith(".mp4")]
if video_files:
    processed = []
    for video_filename in video_files:
        video_name    = os.path.splitext(video_filename)[0]
        meta_csv_path = os.path.join(frames_output_dir, video_name, "frames_labels.csv")
        processed.append(os.path.exists(meta_csv_path))
    if all(processed):
        print("✅ Todos los vídeos ya procesados. Saltando esta celda.")
        sys.exit()  # detiene la ejecución de esta celda


# Crear carpeta de frames si no existe:
if not os.path.exists(frames_output_dir):
    os.makedirs(frames_output_dir)

# Recorre cada vídeo en la carpeta de vídeos:
for video_filename in os.listdir(videos_dir):
    if video_filename.endswith(".mp4"):
        video_path = os.path.join(videos_dir, video_filename)
        # Se asume que el CSV de etiquetas se llama "posesion_<nombre_video>.csv"
        label_filename = "posesion_" + os.path.splitext(video_filename)[0] + ".csv"
        label_path = os.path.join(labels_dir, label_filename)
        if not os.path.exists(label_path):
            print("No se encontró el archivo de etiquetas para", video_filename)
            continue
        
        # Cargar el CSV de etiquetas:
        label_df = pd.read_csv(label_path, sep=';')
        
        # Crear carpeta de salida para los frames de este vídeo:
        video_frames_folder = os.path.join(frames_output_dir, os.path.splitext(video_filename)[0])
        if not os.path.exists(video_frames_folder):
            os.makedirs(video_frames_folder)
        
        # Abrir el vídeo:
        cap = cv2.VideoCapture(video_path)
        frame_idx = 0
        frames_data = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1  # Contador de frames (asumimos que en el CSV los frames empiezan en 1)
            # Buscar la fila del CSV que corresponda a este frame:
            row = label_df[label_df["frame"] == frame_idx]
            if row.empty:
                continue
            # Obtener la etiqueta (el equipo en posesión):
            label = row["team_in_possession"].values[0]
            # Guardar la imagen del frame:
            frame_filename = f"frame_{frame_idx:05d}.jpg"
            frame_path = os.path.join(video_frames_folder, frame_filename)
            cv2.imwrite(frame_path, frame)
            frames_data.append({"frame": frame_idx, "filename": frame_filename, "label": label})
        
        cap.release()
        # Guardar un CSV con la metadata de los frames para este vídeo:
        frames_meta_df = pd.DataFrame(frames_data)
        meta_csv_path = os.path.join(video_frames_folder, "frames_labels.csv")
        frames_meta_df.to_csv(meta_csv_path, index=False)
        print(f"Procesado video: {video_filename} - Frames extraídos: {len(frames_data)}")


✅ Todos los vídeos ya procesados. Saltando esta celda.


SystemExit: 

C:\Users\ANDRES\anaconda3\envs\spacy_env\lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [10]:
import pandas as pd

label_path = r"C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\labels\posesion_D_20220220_1_0000_0030.csv"
label_df = pd.read_csv(label_path)

print(label_df.columns)


Index(['frame;team_in_possession'], dtype='object')


In [28]:
# ────────────────────────────────
#  IMPORTS, PARÁMETROS BÁSICOS Y CACHE DE SECUENCIAS
# ────────────────────────────────
import os
import cv2
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import TimeDistributed, LSTM, Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence

SEQUENCE_LENGTH = 10
IMAGE_SIZE      = (224, 224)
BATCH_SIZE      = 16
DATASET_DIR     = r"C:\Users\ANDRES\Desktop\Master IA\ASIGNATURAS\TFM\CODIGO KAGGLE\archive\top_view\frames"
CACHE_SEQ       = "sequences.npy"

# ────────────────────────────────
#  CARGA/CACHE DE SECUENCIAS
# ────────────────────────────────
if os.path.exists(CACHE_SEQ):
    print("🔄 Cargando secuencias preprocesadas…")
    sequences = np.load(CACHE_SEQ, allow_pickle=True)
else:
    print("⚙️ Preprocesando secuencias desde los frames…")
    def load_dataset_sequences(dataset_dir, sequence_length=SEQUENCE_LENGTH):
        sequences = []
        for video_folder in os.listdir(dataset_dir):
            video_path = os.path.join(dataset_dir, video_folder)
            if not os.path.isdir(video_path):
                continue
            meta_csv = os.path.join(video_path, "frames_labels.csv")
            if not os.path.exists(meta_csv):
                continue
            df = pd.read_csv(meta_csv).dropna(subset=["label"]).sort_values("frame")
            frames = df.to_dict("records")
            for rec in frames:
                rec["video_folder"] = video_folder
            for i in range(0, len(frames) - sequence_length + 1):
                sequences.append(frames[i:i+sequence_length])
        return sequences

    sequences = load_dataset_sequences(DATASET_DIR)
    np.save(CACHE_SEQ, sequences)
    print(f"✅ Secuencias guardadas en {CACHE_SEQ} (total: {len(sequences)})")

# ────────────────────────────────
#  GENERADOR DE DATASETS
# ────────────────────────────────
class FrameSequenceGenerator(Sequence):
    def __init__(self, sequences, dataset_dir, batch_size=BATCH_SIZE,
                 image_size=IMAGE_SIZE, shuffle=True):
        self.sequences   = list(sequences)
        self.batch_size  = batch_size
        self.image_size  = image_size
        self.dataset_dir = dataset_dir
        self.shuffle     = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.sequences) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.sequences)

    def __getitem__(self, index):
        batch = self.sequences[index * self.batch_size:(index + 1) * self.batch_size]
        return self._make_batch(batch)

    def _make_batch(self, batch_sequences):
        batch_X, batch_y = [], []
        for seq in batch_sequences:
            seq_imgs = []
            label    = int(seq[-1]["label"])
            video_folder = seq[-1]["video_folder"]
            video_path   = os.path.join(self.dataset_dir, video_folder)
            for f in seq:
                img_path = os.path.join(video_path, f["filename"])
                img = cv2.imread(img_path)
                if img is None: continue
                img = cv2.resize(img, self.image_size)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
                seq_imgs.append(img)
            if len(seq_imgs) == SEQUENCE_LENGTH:
                batch_X.append(np.array(seq_imgs))
                batch_y.append(label)
        return np.array(batch_X), np.array(batch_y)

# ────────────────────────────────
#  DEFINICIÓN DEL MODELO
# ────────────────────────────────
def build_model(sequence_length=SEQUENCE_LENGTH, image_size=IMAGE_SIZE):
    inputs     = Input(shape=(sequence_length, image_size[0], image_size[1], 3))
    base_model = MobileNetV2(include_top=False, weights='imagenet',
                             input_shape=(image_size[0], image_size[1], 3))
    base_model.trainable = False
    x = TimeDistributed(base_model)(inputs)
    x = TimeDistributed(GlobalAveragePooling2D())(x)
    x = LSTM(64, return_sequences=False)(x)
    outputs = Dense(2, activation='softmax')(x)
    model = Model(inputs, outputs)
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# ────────────────────────────────
#  CREACIÓN DE GENERADORES
# ────────────────────────────────
train_seq, val_seq = train_test_split(sequences, test_size=0.5, random_state=42)
train_gen = FrameSequenceGenerator(train_seq, DATASET_DIR, shuffle=True)
val_gen   = FrameSequenceGenerator(val_seq,   DATASET_DIR, shuffle=False)


🔄 Cargando secuencias preprocesadas…


In [34]:
import os
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

MODEL_PATH      = "ball_possession_model.keras"   # nombre/ubicación del archivo
EPOCHS_HEAD     = 1 #3                               # epochs para entrenar solo la “cabeza”
EPOCHS_TUNING   = 1 #2                               # epochs para el fine-tuning
UNFREEZE_LAYERS = 10                              # cuántas capas finales de MobileNetV2 descongelar

if os.path.exists(MODEL_PATH):
    print("➜ Cargando modelo ya entrenado…")
    os.remove("ball_possession_model.keras")
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
else:
    print("➜ No hay modelo guardado: se construye y entrena uno nuevo")
    model = build_model()
    model.summary()

    # 1) Entrena solo la “cabeza” (LSTM + Dense), manteniendo la base MobileNetV2 congelada
    base = model.layers[1].layer   # asumiendo que TimeDistributed está en layers[1]
    for layer in base.layers:
        layer.trainable = False

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    es = EarlyStopping(
        monitor='val_accuracy',
        patience=1,
        restore_best_weights=True
    )

    print(f"🔨 Entrenando cabeza ({EPOCHS_HEAD} epochs)…")
    history_head = model.fit(
        train_gen,
        steps_per_epoch=100,  #QUITAR
        validation_data=val_gen,
        validation_steps=100, #200, QUITAR
        epochs=EPOCHS_HEAD,
        callbacks=[es],
        verbose=2
    )

    # 2) Fine-tuning: descongela las últimas UNFREEZE_LAYERS capas de la base
    print(f"🔧 Fine-tuning: descongelando últimas {UNFREEZE_LAYERS} capas de MobileNetV2")
    for layer in base.layers[-UNFREEZE_LAYERS:]:
        layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"🔨 Entrenando fine-tuning ({EPOCHS_TUNING} epochs)…")
    history_tune = model.fit(
        train_gen,
        steps_per_epoch=50,  #QUITAR
        validation_data=val_gen,
        validation_steps=50, #200,    QUITAR
        epochs=EPOCHS_TUNING,
        callbacks=[es],
        verbose=2
    )

    # 3) Guardar modelo final
    model.save(MODEL_PATH)
    print(f"✅ Modelo guardado en {MODEL_PATH}")


➜ No hay modelo guardado: se construye y entrena uno nuevo
Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_5 (InputLayer)        [(None, 10, 224, 224, 3)  0         
                             ]                                   
                                                                 
 time_distributed_4 (TimeDis  (None, 10, 7, 7, 1280)   2257984   
 tributed)                                                       
                                                                 
 time_distributed_5 (TimeDis  (None, 10, 1280)         0         
 tributed)                                                       
                                                                 
 lstm_2 (LSTM)               (None, 64)                344320    
                                                                 
 dense_2 (Dense)             (None, 2)                 130       


In [ ]:
# Evaluar el modelo en el set de validación:
val_loss, val_accuracy = model.evaluate(val_gen)
print("Pérdida en validación:", val_loss)
print("Precisión en validación:", val_accuracy)

import matplotlib.pyplot as plt

# Tomar una secuencia de validación
X_val, y_val = val_gen[0]
y_pred = np.argmax(model.predict(X_val), axis=1)

plt.figure(figsize=(8,4))
plt.plot(y_val, 'bo-', label="Etiqueta Real")
plt.plot(y_pred, 'ro-', label="Predicción")
plt.xlabel("Índice de secuencia")
plt.ylabel("Equipo en posesión")
plt.legend()
plt.title("Comparación Etiqueta Real vs. Predicción (Primer Batch de Validación)")
plt.show()

In [ ]:
import cv2
import numpy as np

SEQUENCE_LENGTH = 10
IMAGE_SIZE = (224, 224)

def video_to_sequences(video_path, sequence_length=SEQUENCE_LENGTH):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok: break
        frame = cv2.resize(frame, IMAGE_SIZE)
        frame = frame.astype("float32") / 255.0
        frames.append(frame)
    cap.release()

    # Agrupamos en secuencias deslizantes de N frames
    seqs = []
    for i in range(len(frames) - sequence_length + 1):
        seq = frames[i:i + sequence_length]
        seqs.append(seq)
    return np.array(seqs)

def predict_possession(video_path):
    sequences = video_to_sequences(video_path)
    preds = model.predict(sequences)              # (num_seq, 2)
    labels = np.argmax(preds, axis=1)             # 0 ó 1 por secuencia

    # Ejemplo: mayoría simple para “quién posee”
    overall = int(np.round(labels.mean()))
    return overall, labels            # (global, frame‑a‑frame)

# Uso:
video_path = "mi_video.mp4"
global_label, frame_labels = predict_possession(video_path)
print(f"Equipo en posesión (global): {global_label}")